# DA-Fusion Pipeline (ResNet50 + Ridge)

Aligned with `src/shared/works_with_kaggle.ipynb` so local results are directly comparable to Kaggle scores. Compares three augmentation strategies with the same ResNet50 extractor and Ridge regression head:
- **Baseline**: Real images only
- **RandAugment**: Real images with classical augmentation applied before ResNet50 extraction
- **DA-Fusion**: Real + diffusion-generated synthetic images (from `generate_augmented.py`)

## Imports and config

In [1]:
import os, sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms

from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor

REPO_ROOT = Path(r"C:\Users\Eier\OneDrive\Maskinlæring\INF367A\CSIRO---Image2Biomass-Prediction")

TARGETS = ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]
WEIGHTS = {
    "Dry_Green_g":  0.1,
    "Dry_Dead_g":   0.1,
    "Dry_Clover_g": 0.1,
    "GDM_g":        0.2,
    "Dry_Total_g":  0.5,
}

CSV_PATH      = REPO_ROOT / "data" / "tabular" / "train" / "train.csv"
AUG_CSV       = REPO_ROOT / "data" / "tabular" / "train" / "train_augmented.csv"
TRAIN_IMG_DIR = REPO_ROOT / "data" / "image" / "train"
IMAGE_DIR     = REPO_ROOT / "data" / "image"   # synthetic images live under subdirs

# Canonical size used for ALL images before splitting into halves. Real images
# are already 2000x1000; synthetic images (square 512x512 from Stable Diffusion)
# get resized to this canvas first so each half has the same aspect ratio as the
# real halves. This keeps the ResNet50 feature distribution consistent between
# real and synthetic.
CANONICAL_SIZE = (2000, 1000)   # (W, H)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
print("train.csv     :", CSV_PATH.exists())
print("train img dir :", TRAIN_IMG_DIR.exists())
print("aug csv       :", AUG_CSV.exists())


# ---------- Helpers (mirror works_with_kaggle.ipynb) ----------
def load_train_data(csv_path):
    df = pd.read_csv(csv_path)
    df["image_id"] = df["sample_id"].str.split("__").str[0]
    df_wide = df.pivot_table(
        index=["image_id", "image_path"],
        columns="target_name",
        values="target",
    ).reset_index()
    y_values = df_wide[TARGETS].values
    return df_wide, y_values


def weighted_global_r2(y_true, y_pred):
    w  = np.array([WEIGHTS[t] for t in TARGETS], dtype=np.float64)
    yt = y_true.reshape(-1)
    yp = y_pred.reshape(-1)
    ww = np.repeat(w, y_true.shape[0])
    ybar   = np.sum(ww * yt) / np.sum(ww)
    ss_res = np.sum(ww * (yt - yp) ** 2)
    ss_tot = np.sum(ww * (yt - ybar) ** 2) + 1e-12
    return float(1.0 - ss_res / ss_tot)


def rmse_per_target(y_true, y_pred):
    return {
        t: float(np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i])))
        for i, t in enumerate(TARGETS)
    }


# ---------- ResNet50 extractor (matches works_with_kaggle.ipynb) ----------
class ResNet50Extractor:
    """Pretrained ResNet50 (falls back to random init if no internet).
       Every image is first resized to CANONICAL_SIZE (2000x1000), then split
       left/right into 1000x1000 squares. Each half is ResNet50'd to 2048-d
       avg-pooled features, then concatenated to 4096-d. Using a canonical
       canvas guarantees real (2000x1000) and synthetic (e.g. 512x512)
       images produce features in the same distribution."""
    def __init__(self, device="cpu", transform=None):
        self.device = device
        try:
            weights = models.ResNet50_Weights.DEFAULT
            resnet  = models.resnet50(weights=weights)
            print("Loaded pretrained ResNet50 weights.")
        except Exception as e:
            print("Could not load pretrained weights, falling back to random init:", e)
            resnet = models.resnet50(weights=None)

        self.model = nn.Sequential(*list(resnet.children())[:-1]).to(device).eval()
        self.preprocess = transform or transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std =[0.229, 0.224, 0.225]),
        ])

    @torch.no_grad()
    def _one(self, pil_img):
        t = self.preprocess(pil_img).unsqueeze(0).to(self.device)
        return self.model(t).flatten().cpu().numpy().astype(np.float32)

    def get_features(self, image_path):
        if not os.path.exists(image_path):
            print("Missing:", image_path)
            return np.zeros(4096, dtype=np.float32)
        try:
            img = Image.open(image_path).convert("RGB")
            # Normalise to canonical canvas before splitting
            if img.size != CANONICAL_SIZE:
                img = img.resize(CANONICAL_SIZE, Image.BILINEAR)
            W, H = img.size            # (2000, 1000)
            left  = img.crop((0,     0, W // 2, H))   # 1000x1000
            right = img.crop((W // 2, 0, W,     H))   # 1000x1000
            return np.concatenate([self._one(left), self._one(right)]).astype(np.float32)
        except Exception as e:
            print(f"Error {image_path}: {e}")
            return np.zeros(4096, dtype=np.float32)

Device: cpu
train.csv     : True
train img dir : True
aug csv       : True


## Extract real-image features and build train/val split

ResNet50 features are extracted once and cached to disk; subsequent runs reload from cache. Train/val split uses the same `train_test_split(test_size=0.2, random_state=42)` as `works_with_kaggle.ipynb`.

In [2]:
CACHE_DIR = REPO_ROOT / "src" / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

REAL_FEATS_CACHE = CACHE_DIR / "features_resnet50.npy"
REAL_IDS_CACHE   = CACHE_DIR / "image_ids_resnet50.npy"

# ---------- Load label wide-frame ----------
train_df, _ = load_train_data(CSV_PATH)
print(f"Train images: {len(train_df)}")

# ---------- Extract real features (cached) ----------
extractor = ResNet50Extractor(device=device)

if REAL_FEATS_CACHE.exists() and REAL_IDS_CACHE.exists():
    X_real      = np.load(REAL_FEATS_CACHE).astype(np.float32)
    real_ids_np = np.load(REAL_IDS_CACHE, allow_pickle=True)
    print(f"Loaded cached real features: {X_real.shape}")
else:
    feats, ids = [], []
    for i, row in enumerate(train_df.itertuples(index=False), 1):
        img_path = TRAIN_IMG_DIR / f"{row.image_id}.jpg"
        feats.append(extractor.get_features(str(img_path)))
        ids.append(row.image_id)
        if i % 50 == 0:
            print(f"  {i}/{len(train_df)}")
    X_real      = np.asarray(feats, dtype=np.float32)
    real_ids_np = np.asarray(ids)
    np.save(REAL_FEATS_CACHE, X_real)
    np.save(REAL_IDS_CACHE,   real_ids_np)
    print(f"Saved real feature cache: {X_real.shape}")

# align labels to the order used during extraction
id_to_label = {row.image_id: train_df.loc[train_df["image_id"] == row.image_id, TARGETS].values[0]
               for row in train_df.itertuples(index=False)}
Y_real = np.stack([id_to_label[iid] for iid in real_ids_np]).astype(np.float32)

# ---------- Fixed train/val split ----------
X_tr, X_val, y_tr, y_val, ids_tr, ids_val = train_test_split(
    X_real, Y_real, real_ids_np,
    test_size=0.2, random_state=42,
)
print(f"Train: {X_tr.shape}  Val: {X_val.shape}")

Train images: 357
Loaded pretrained ResNet50 weights.
  50/357
  100/357
  150/357
  200/357
  250/357
  300/357
  350/357
Saved real feature cache: (357, 4096)
Train: (285, 4096)  Val: (72, 4096)


## Experiment 1: Baseline (real data only)

Fit `Ridge(alpha=10.0)` per target on the real-image training split. This matches `works_with_kaggle.ipynb` and serves as the reference score.

In [3]:
print("=== EXPERIMENT 1: Baseline (real data only) ===")

model_1 = MultiOutputRegressor(Ridge(alpha=10.0, random_state=42))
model_1.fit(X_tr, y_tr)
pred_1 = model_1.predict(X_val)

r2_1   = weighted_global_r2(y_val, pred_1)
rmse_1 = rmse_per_target(y_val, pred_1)
print(f"Baseline  val weighted R² = {r2_1:.4f}")
print(f"          val overall  R² = {r2_score(y_val, pred_1, multioutput='uniform_average'):.4f}")
for t in TARGETS:
    print(f"  RMSE {t:<18}: {rmse_1[t]:.4f}")

=== EXPERIMENT 1: Baseline (real data only) ===
Baseline  val weighted R² = 0.5526
          val overall  R² = 0.4349
  RMSE Dry_Green_g       : 14.6882
  RMSE Dry_Dead_g        : 11.1141
  RMSE Dry_Clover_g      : 9.3022
  RMSE GDM_g             : 15.6202
  RMSE Dry_Total_g       : 17.4015


## Experiment 2: RandAugment baseline

Re-extract features from the training images with classical `RandAugment` applied inside the ResNet50 preprocessing pipeline. Provides a comparison between traditional pixel-space augmentation and diffusion-based DA-Fusion.

In [4]:
print("=== EXPERIMENT 2: RandAugment baseline ===")

# RandAugment preprocessing: apply RandAugment on the cropped half, then ResNet50 transform
_ra_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

# A second extractor sharing ResNet50 weights but with RandAugment preprocessing.
# (We keep the same ResNet50 backbone; only the input transform changes.)
torch.manual_seed(42); np.random.seed(42)
ra_extractor = ResNet50Extractor(device=device, transform=_ra_transform)

# Extract only over the training split (validation set stays clean)
ra_feats = []
for i, iid in enumerate(ids_tr, 1):
    img_path = TRAIN_IMG_DIR / f"{iid}.jpg"
    ra_feats.append(ra_extractor.get_features(str(img_path)))
    if i % 50 == 0:
        print(f"  {i}/{len(ids_tr)}")
X_ra_tr = np.asarray(ra_feats, dtype=np.float32)
print("X_ra_tr:", X_ra_tr.shape)

model_2 = MultiOutputRegressor(Ridge(alpha=10.0, random_state=42))
model_2.fit(X_ra_tr, y_tr)
pred_2 = model_2.predict(X_val)   # validation features are still clean (no RandAugment)

r2_2   = weighted_global_r2(y_val, pred_2)
rmse_2 = rmse_per_target(y_val, pred_2)
print(f"RandAugment  val weighted R² = {r2_2:.4f}")
for t in TARGETS:
    print(f"  RMSE {t:<18}: {rmse_2[t]:.4f}")

=== EXPERIMENT 2: RandAugment baseline ===
Loaded pretrained ResNet50 weights.
  50/285
  100/285
  150/285
  200/285
  250/285
X_ra_tr: (285, 4096)
RandAugment  val weighted R² = 0.5859
  RMSE Dry_Green_g       : 14.7895
  RMSE Dry_Dead_g        : 11.5029
  RMSE Dry_Clover_g      : 9.1815
  RMSE GDM_g             : 15.0402
  RMSE Dry_Total_g       : 16.9568


## Experiment 3: DA-Fusion (real + synthetic)

Combine real training features with ResNet50 features of the DA-Fusion synthetic images. Synthetic features are extracted once and cached. Validation is on real images only.

To prevent leakage, any synthetic image whose `source_image` is in the validation split is excluded.

**Requires**: `generate_augmented.py` must have been run to create `train_augmented.csv` and the synthetic image files under `data/image/`.

In [5]:
AUG_FEATS_CACHE = CACHE_DIR / "features_resnet50_aug.npy"
AUG_IDS_CACHE   = CACHE_DIR / "image_ids_resnet50_aug.npy"

if not AUG_CSV.exists():
    print("[SKIP] train_augmented.csv not found — run generate_augmented.py first.")
else:
    print("=== EXPERIMENT 3: DA-Fusion (real + synthetic) ===")

    df_aug   = pd.read_csv(AUG_CSV)
    df_synth = df_aug[df_aug["is_synthetic"].astype(bool)].copy()
    print(f"Synthetic rows in CSV: {len(df_synth)}")

    # Leakage filter: keep only synthetics whose source image is in the train split
    def _source_id(p):
        return Path(p).stem  # 'ID1234' from 'train/ID1234.jpg'
    train_source_ids = set(ids_tr)
    if "source_image" in df_synth.columns:
        df_synth = df_synth[
            df_synth["source_image"].apply(_source_id).isin(train_source_ids)
        ].reset_index(drop=True)
        print(f"After leakage filter (source in train split): {len(df_synth)}")

    # ---------- Extract synthetic features (cached) ----------
    if AUG_FEATS_CACHE.exists() and AUG_IDS_CACHE.exists():
        X_aug      = np.load(AUG_FEATS_CACHE).astype(np.float32)
        aug_ids_np = np.load(AUG_IDS_CACHE, allow_pickle=True)
        print(f"Loaded cached synthetic features: {X_aug.shape}")
        # align to the (possibly filtered) subset using stem-id
        filtered_stems = df_synth["image_path"].apply(lambda p: Path(p).stem).values
        keep_mask = np.isin(aug_ids_np, filtered_stems)
        X_aug      = X_aug[keep_mask]
        aug_ids_np = aug_ids_np[keep_mask]
        print(f"  After filter alignment: {X_aug.shape}")
    else:
        feats, ids = [], []
        for i, row in enumerate(df_synth.itertuples(index=False), 1):
            img_path = IMAGE_DIR / row.image_path   # e.g. augmented/.../IDxxx_aug0.jpg
            feats.append(extractor.get_features(str(img_path)))
            ids.append(Path(row.image_path).stem)
            if i % 50 == 0:
                print(f"  {i}/{len(df_synth)}")
        X_aug      = np.asarray(feats, dtype=np.float32)
        aug_ids_np = np.asarray(ids)
        np.save(AUG_FEATS_CACHE, X_aug)
        np.save(AUG_IDS_CACHE,   aug_ids_np)
        print(f"Saved synthetic cache: {X_aug.shape}")

    # ---------- Labels for synthetic images (copied from source row) ----------
    stem_to_label = {
        Path(row["image_path"]).stem: row[TARGETS].values.astype(np.float32)
        for _, row in df_synth.iterrows()
    }
    Y_aug = np.stack([stem_to_label[iid] for iid in aug_ids_np]).astype(np.float32)

    # ---------- Combine real-train + synthetic and refit Ridge ----------
    X_daf_tr = np.vstack([X_tr, X_aug])
    y_daf_tr = np.vstack([y_tr, Y_aug])
    print(f"Combined train: {X_daf_tr.shape}  (real={len(X_tr)}, synth={len(X_aug)})")

    model_3 = MultiOutputRegressor(Ridge(alpha=10.0, random_state=42))
    model_3.fit(X_daf_tr, y_daf_tr)
    pred_3 = model_3.predict(X_val)

    r2_3   = weighted_global_r2(y_val, pred_3)
    rmse_3 = rmse_per_target(y_val, pred_3)
    print(f"DA-Fusion  val weighted R² = {r2_3:.4f}")
    for t in TARGETS:
        print(f"  RMSE {t:<18}: {rmse_3[t]:.4f}")

=== EXPERIMENT 3: DA-Fusion (real + synthetic) ===
Synthetic rows in CSV: 1071
After leakage filter (source in train split): 855
  50/855
  100/855
  150/855
  200/855
  250/855
  300/855
  350/855
  400/855
  450/855
  500/855
  550/855
  600/855
  650/855
  700/855
  750/855
  800/855
  850/855
Saved synthetic cache: (855, 4096)
Combined train: (1140, 4096)  (real=285, synth=855)
DA-Fusion  val weighted R² = 0.4613
  RMSE Dry_Green_g       : 16.4951
  RMSE Dry_Dead_g        : 11.9836
  RMSE Dry_Clover_g      : 9.7327
  RMSE GDM_g             : 15.0713
  RMSE Dry_Total_g       : 18.5075


## Results Summary

In [6]:
header = f"{'Method':<16} | {'Val R\u00b2':>6} | " + " | ".join(f"{t.split('_')[1]:>8}" for t in TARGETS)
sep = "-" * len(header)
print(sep)
print(header)
print(sep)

rows = [("Baseline", r2_1, rmse_1), ("RandAugment", r2_2, rmse_2)]
if "r2_3" in globals():
    rows.append(("DA-Fusion", r2_3, rmse_3))

for name, r2, rmse in rows:
    rmse_vals = " | ".join(f"{rmse[t]:>8.2f}" for t in TARGETS)
    print(f"{name:<16} | {r2:>6.4f} | {rmse_vals}")
print(sep)

--------------------------------------------------------------------------------
Method           | Val R² |    Green |     Dead |   Clover |        g |    Total
--------------------------------------------------------------------------------
Baseline         | 0.5526 |    14.69 |    11.11 |     9.30 |    15.62 |    17.40
RandAugment      | 0.5859 |    14.79 |    11.50 |     9.18 |    15.04 |    16.96
DA-Fusion        | 0.4613 |    16.50 |    11.98 |     9.73 |    15.07 |    18.51
--------------------------------------------------------------------------------
